In [1]:
import json
import pathlib
import random

In [2]:
from google.colab import drive
drive.mount('/gdrive')

Mounted at /gdrive


In [3]:
# 1. Define your exact folders
image_folder = pathlib.Path("/gdrive/MyDrive/data")
json_folder = pathlib.Path("/gdrive/MyDrive/menu_json_ground_truth")

# Create a folder specifically for the LLaMA-Factory formatted data
output_base = pathlib.Path("/gdrive/MyDrive/llamafactory-ocr-finetune-data")
output_base.mkdir(parents=True, exist_ok=True)

In [4]:
# 2. Gather and format the dataset
dataset = []
valid_extensions = {'.jpg', '.jpeg', '.png', '.webp'}

# prompt
task_message = """<image>
Extract every food or drink item and its price from this menu image.
Keep item names and prices exactly as written in the image.
If an item has multiple sizes, list each size as a separate entry.
Only include items that have a visible price. Skip anything without a price.
Return a JSON object with one key "items" containing an array. Each item has "name" (string) and "price" (string).""".strip()


In [5]:
for img_path in image_folder.iterdir():
    if img_path.suffix.lower() not in valid_extensions:
        continue

    # Find the corresponding JSON file
    json_path = json_folder / f"{img_path.stem}.json"

    if not json_path.exists():
        continue # Skip if this image somehow doesn't have a JSON

    # Load the ground truth JSON
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            ground_truth = json.load(f)
    except Exception as e:
        print(f"Error reading {json_path}: {e}")
        continue

    # 3. Build the ShareGPT format record required by LLaMA-Factory
    record = {
        "conversations": [
            {
                "from": "human",
                "value": task_message
            },
            {
                "from": "gpt",
                # Convert the JSON dictionary back to a formatted string for the model to learn
                "value": json.dumps(ground_truth, ensure_ascii=False, indent=2)
            }
        ],
        "images": [
            str(img_path) # Absolute path to the image
        ]
    }
    dataset.append(record)

In [6]:
# 4. Split into Train and Validation sets
# Shuffle with a fixed seed so your train/val split is always the same if you run it again
random.Random(42).shuffle(dataset)

# 90% for training, 10% for validation
split_idx = int(len(dataset) * 0.9)
train_ds = dataset[:split_idx]
val_ds = dataset[split_idx:]

print(f"Total image/JSON pairs matched: {len(dataset)}")
print(f"Training set size: {len(train_ds)}")
print(f"Validation set size: {len(val_ds)}")


Total image/JSON pairs matched: 3080
Training set size: 2772
Validation set size: 308


In [7]:
# 5. Save the final datasets
train_file_path = output_base / "train.json"
val_file_path = output_base / "val.json"

with open(train_file_path, "w", encoding="utf-8") as dest:
    json.dump(train_ds, dest, ensure_ascii=False, indent=2)

with open(val_file_path, "w", encoding="utf-8") as dest:
    json.dump(val_ds, dest, ensure_ascii=False, indent=2)

print(f"\nSaved formatting!")
print(f"Train path: {train_file_path}")
print(f"Val path: {val_file_path}")



Saved formatting!
Train path: /gdrive/MyDrive/llamafactory-ocr-finetune-data/train.json
Val path: /gdrive/MyDrive/llamafactory-ocr-finetune-data/val.json


In [8]:
# Print the very first record in the training dataset
print(json.dumps(train_ds[0], indent=2, ensure_ascii=False))

{
  "conversations": [
    {
      "from": "human",
      "value": "<image>\nExtract every food or drink item and its price from this menu image.\nKeep item names and prices exactly as written in the image.\nIf an item has multiple sizes, list each size as a separate entry.\nOnly include items that have a visible price. Skip anything without a price.\nReturn a JSON object with one key \"items\" containing an array. Each item has \"name\" (string) and \"price\" (string)."
    },
    {
      "from": "gpt",
      "value": "{\n  \"items\": [\n    {\n      \"name\": \"سلطة ثومية\",\n      \"price\": \"5\"\n    },\n    {\n      \"name\": \"سلطة بنجر\",\n      \"price\": \"7\"\n    },\n    {\n      \"name\": \"مخلل\",\n      \"price\": \"5\"\n    },\n    {\n      \"name\": \"جرجير\",\n      \"price\": \"5\"\n    },\n    {\n      \"name\": \"سلطة بابا غنوج\",\n      \"price\": \"5\"\n    },\n    {\n      \"name\": \"سلطة متبل\",\n      \"price\": \"5\"\n    },\n    {\n      \"name\": \"سلطة كو

### Clean corrupted files

In [9]:
import os

broken_files = ["menu_70_5", "menu_79_13"]
base_paths = [
    "/gdrive/MyDrive/data/",
    "/gdrive/MyDrive/menu_json_ground_truth/"
]

print("Executing targeted clean-up...")

for file_stem in broken_files:
    # Delete the Image
    img_path = f"{base_paths[0]}{file_stem}.jpg"
    if os.path.exists(img_path):
        os.remove(img_path)
        print(f"🗑️ Deleted: {img_path}")

    # Delete the empty JSON
    json_path = f"{base_paths[1]}{file_stem}.json"
    if os.path.exists(json_path):
        os.remove(json_path)
        print(f"🗑️ Deleted: {json_path}")

print("✅ Clean-up complete!")

Executing targeted clean-up...
✅ Clean-up complete!


## Push dataset to **hf**

In [10]:
!pip install -q huggingface_hub


In [11]:
from huggingface_hub import HfApi, login
import os
from google.colab import userdata

# 1. Authenticate (Paste your HF Write Token here)
HF_TOKEN = userdata.get('huggingfacetoken')
login(token=HF_TOKEN)

api = HfApi()

In [12]:
DATASET_REPO_ID = "mohamedashraff22/arabic-menus-dataset"

print(f"Creating dataset repository: {DATASET_REPO_ID}...")
api.create_repo(repo_id=DATASET_REPO_ID, repo_type="dataset", private=True, exist_ok=True)

# 2. Upload the formatted JSON files
print("Uploading train.json...")
api.upload_file(
    path_or_fileobj="/gdrive/MyDrive/llamafactory-ocr-finetune-data/train.json",
    path_in_repo="train.json",
    repo_id=DATASET_REPO_ID,
    repo_type="dataset"
)

print("Uploading val.json...")
api.upload_file(
    path_or_fileobj="/gdrive/MyDrive/llamafactory-ocr-finetune-data/val.json",
    path_in_repo="val.json",
    repo_id=DATASET_REPO_ID,
    repo_type="dataset"
)

# 3. Upload the 3,400+ images
print("Uploading the massive image folder (This will take a few minutes)...")
api.upload_folder(
    folder_path="/gdrive/MyDrive/data",
    path_in_repo="images", # This will put them all neatly into an 'images' folder on HF
    repo_id=DATASET_REPO_ID,
    repo_type="dataset",
    allow_patterns=["*.jpg", "*.jpeg", "*.png", "*.webp"] # Only grabs images, ignores Google Drive hidden files
)

print(f"\n✅ Upload complete! Your 3.4k dataset is safely locked in at: https://huggingface.co/datasets/{DATASET_REPO_ID}")

Creating dataset repository: mohamedashraff22/arabic-menus-dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Uploading train.json...
Uploading val.json...
Uploading the massive image folder (This will take a few minutes)...


It seems you are trying to upload a large folder at once. This might take some time and then fail if the folder is too large. For such cases, it is recommended to upload in smaller batches or to use `HfApi().upload_large_folder(...)`/`hf upload-large-folder` instead. For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/upload#upload-a-large-folder.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

...

  /gdrive/MyDrive/data/005.jpg: 100%|##########|  418kB /  418kB            

  /gdrive/MyDrive/data/001.jpg: 100%|##########|  663kB /  663kB            

  ...2466189142453308257_n.jpg: 100%|##########| 63.9kB / 63.9kB            

  ...3286200802145416343_n.jpg: 100%|##########| 68.3kB / 68.3kB            


✅ Upload complete! Your 3.4k dataset is safely locked in at: https://huggingface.co/datasets/mohamedashraff22/arabic-menus-dataset
